# 🔑 Solutions: Lesson 16 · Capstone · FieldAdvisor

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

In [2]:
# ══════════════ GIVEN: price-forecasting data preparation (you built this in Lesson 12) ══════════════
import pandas as pd
W, H = 52, 4                                                     # 52-week input window → forecast 1..4 weeks ahead
markets = ["Nairobi", "Eldoret", "Kisumu"]
_all = dl.load_prices(None)
series = {}
for _m in markets:
    _df = _all[_all.market == _m].reset_index(drop=True); _doy = _df.date.dt.dayofyear.values
    series[_m] = dict(logp=np.log(_df.price_kes_90kg.values.astype(float)), price=_df.price_kes_90kg.values.astype(float),
                      rain=_df.rainfall_mm.values, fuel=_df.fuel_kes_per_litre.values,
                      sin=np.sin(2 * np.pi * _doy / 365.25), cos=np.cos(2 * np.pi * _doy / 365.25))
dates = pd.to_datetime(_all[_all.market == "Nairobi"].date.values); n = len(dates)
t_tr = int(np.where(dates <= "2020-12-31")[0].max()); t_va = int(np.where(dates <= "2022-12-31")[0].max())
rain_mean = np.mean([series[m]["rain"][:t_tr + 1].mean() for m in markets])      # TRAIN-only statistic

def build(market, split, multi=True):
    """X (N, 52, 5): relative log-price, rain, relative fuel, sin, cos · Y (N, 4): log-changes at +1..+4 weeks · origins (N,)."""
    d = series[market]; lo, hi = {"train": (W - 1, t_tr), "val": (t_tr, t_va), "test": (t_va, n - 1)}[split]
    X, Y, origins = [], [], []
    for t in range(lo, hi - H + 1):
        sl = slice(t - W + 1, t + 1)
        feats = [d["logp"][sl] - d["logp"][t]]
        feats += [d["rain"][sl] / rain_mean, d["fuel"][sl] / d["fuel"][t] - 1, d["sin"][sl], d["cos"][sl]] if multi else [d["sin"][sl], d["cos"][sl]]
        X.append(np.stack(feats, -1)); Y.append(d["logp"][t + 1:t + H + 1] - d["logp"][t]); origins.append(t)
    return np.array(X, np.float32), np.array(Y, np.float32), np.array(origins)

def pooled(split, mkts, multi=True):
    parts = [build(m, split, multi) for m in mkts]
    return np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])

def to_kes(origins, delta, market="Nairobi"):
    return series[market]["price"][origins][:, None] * np.exp(delta)

X_va, Y_va, origins_va = build("Nairobi", "val")
X_te, Y_te, origins_te = build("Nairobi", "test")
print(f"price windows → train/val/test (Nairobi): {len(build('Nairobi', 'train')[1])} / {len(Y_va)} / {len(Y_te)}  | inputs {X_te.shape[1:]} | targets {Y_te.shape[1:]}")

def format_card(card):
    """Pretty-print an advice card (a dict) as a farmer-friendly text box."""
    lines = ["┌" + "─" * 66 + "┐", "│ 🌽 FIELD ADVISOR".ljust(67) + "│", "├" + "─" * 66 + "┤"]
    for key, label in [("diagnosis_text", "🍃 Leaf"), ("price_text", "📈 Price"), ("action", "✅ Action")]:
        txt = str(card.get(key, ""))
        while txt:
            lines.append(f"│ {label if txt == str(card.get(key, '')) else ' ' * len(label)}: {txt[:52]}".ljust(67) + "│"); txt = txt[52:]
    lines.append("└" + "─" * 66 + "┘")
    return "\n".join(lines)

price windows → train/val/test (Nairobi): 467 / 101 / 154  | inputs (52, 5) | targets (4,)


---
### 🧩 Problem 16.1: Milestone 1 - Leaf data pipeline  &nbsp; `⭐`

Write `make_leaf_data(batch_size=64)` that loads the maize-leaf dataset and returns a dict with:

* `train_dl`: a shuffled `DataLoader` of normalised **channels-first** training images and labels,
* `val_dl`: a `DataLoader` for validation (batch 450),
* `X_tr`, `X_te`: the normalised training / test image tensors `(N, 3, 48, 48)`,
* `y_te`: test labels (NumPy), and `class_names`.

Normalise **per colour channel using TRAIN statistics only** (Lesson 10). Then call `leaf = make_leaf_data()`.


In [3]:
def make_leaf_data(batch_size=64):
    d = dl.load_leaves()
    xt = d["x_train"].astype(np.float64)                      # float64 for accurate statistics
    mean, std = xt.mean(axis=(0, 1, 2)), xt.std(axis=(0, 1, 2))
    prep = lambda a: torch.tensor(((a - mean) / std).transpose(0, 3, 1, 2), dtype=torch.float32)
    X_tr, X_val, X_te = prep(d["x_train"]), prep(d["x_val"]), prep(d["x_test"])
    return dict(train_dl=DataLoader(TensorDataset(X_tr, torch.tensor(d["y_train"])), batch_size=batch_size, shuffle=True),
                val_dl=DataLoader(TensorDataset(X_val, torch.tensor(d["y_val"])), batch_size=450),
                X_tr=X_tr, X_te=X_te, y_te=d["y_test"], class_names=dl.LEAF_CLASSES)

leaf = make_leaf_data()

In [4]:
# ▶ self-check
ok = "leaf" in globals() and isinstance(leaf, dict)
dl.check("returns a dict with the required keys", ok and all(k in leaf for k in ("train_dl", "val_dl", "X_tr", "X_te", "y_te", "class_names")))
dl.check("channels-first test tensor (450, 3, 48, 48)", ok and tuple(leaf["X_te"].shape) == (450, 3, 48, 48))
dl.check("training set is standardised per channel (mean≈0, std≈1)", ok and torch.allclose(leaf["X_tr"].mean((0, 2, 3)), torch.zeros(3), atol=1e-3) and torch.allclose(leaf["X_tr"].std((0, 2, 3)), torch.ones(3), atol=2e-2))
dl.check("test set uses TRAIN statistics (its mean is not exactly 0)", ok and leaf["X_te"].mean((0, 2, 3)).abs().max() > 1e-6)
dl.check("5 class names", ok and len(leaf["class_names"]) == 5)

✅ returns a dict with the required keys
✅ channels-first test tensor (450, 3, 48, 48)
✅ training set is standardised per channel (mean≈0, std≈1)
✅ test set uses TRAIN statistics (its mean is not exactly 0)
✅ 5 class names


---
### 🧩 Problem 16.2: Milestone 2 - The disease classifier  &nbsp; `⭐⭐⭐`

Write `train_leaf_model(leaf, epochs=25)` that builds and **trains a CNN** on `leaf["train_dl"]` (validating on `leaf["val_dl"]`) and returns the trained model.
Target: **≥ 85 % test accuracy** and **recall ≥ 65 % on every class** (aim for 70 %+). Use what you learned in Projects 2–3: conv blocks with BatchNorm, dropout, **label-preserving augmentation** (rotations by 90° and flips), Adam with lr ≈ 2e-3.
The starter is a weak linear placeholder: replace it. *(Training takes ~1.5 minutes on a CPU.)*


In [5]:
class AugLoader:
    def __init__(self, loader): self.loader = loader
    def __len__(self): return len(self.loader)
    def __iter__(self):
        for xb, yb in self.loader:
            out = []
            for x in xb:
                x = torch.rot90(x, int(torch.randint(0, 4, (1,))), dims=(1, 2))
                out.append(x.flip(2) if torch.rand(1) < 0.5 else x)
            yield torch.stack(out), yb

def train_leaf_model(leaf, epochs=25):
    block = lambda a, b: nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))
    dl.set_seed(0)
    model = nn.Sequential(block(3, 16), block(16, 32), block(32, 64), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 6 * 6, 5))
    dl.fit(model, AugLoader(leaf["train_dl"]), leaf["val_dl"], nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), 2e-3),
           epochs=epochs, device=cpu, verbose=False)
    return model

leaf_model = train_leaf_model(leaf)

In [6]:
# ▶ self-check
probs = dl.predict_proba(leaf_model, leaf["X_te"], device=cpu).numpy(); preds = probs.argmax(1); y = leaf["y_te"]
acc = (preds == y).mean(); recalls = [(preds[y == c] == c).mean() for c in range(5)]
print("per-class recall:", {n: round(float(r), 2) for n, r in zip(leaf["class_names"], recalls)})
dl.check(f"test accuracy ≥ 85%  (yours: {acc:.1%})", acc >= 0.85, "add augmentation + a 3-block CNN")
dl.check(f"every class recall ≥ 65%  (worst: {min(recalls):.1%})", min(recalls) >= 0.65)
dl.check("model outputs 5 logits", tuple(leaf_model(leaf['X_te'][:2]).shape) == (2, 5))

per-class recall: {'healthy': 0.92, 'gray_leaf_spot': 0.91, 'northern_leaf_blight': 0.86, 'common_rust': 1.0, 'maize_streak': 0.88}
✅ test accuracy ≥ 85%  (yours: 91.1%)
✅ every class recall ≥ 65%  (worst: 86.0%)
✅ model outputs 5 logits


**Why this works.** Rotation/flip augmentation gives the network 8× more orientations to learn from; BatchNorm and dropout regularise a model trained on only 2,100 images.

---
### 🧩 Problem 16.3: Milestone 3 - The price forecaster  &nbsp; `⭐⭐⭐`

Write `forecast()` returning a dict `{"val": ..., "test": ...}` of predicted **log-changes** (shape `(N, 4)`, for +1..+4 weeks) for Nairobi's validation and test windows (`X_va`/`X_te` are given).
Target: **4-week test MAE ≤ 0.8 × the naive forecast's** (naive = zeros).
The starter returns the naive forecast. Options: pick one and justify it in your notes:
**(a)** Ridge regression on the flattened window; **(b)** an LSTM/GRU (Lesson 12), ideally pooled over the three markets and averaged over seeds.


In [7]:
class Forecaster(nn.Module):
    def __init__(self, n_feat=5, hidden=32, drop=0.2):
        super().__init__()
        self.rnn = nn.LSTM(n_feat, hidden, batch_first=True); self.drop = nn.Dropout(drop); self.head = nn.Linear(hidden, H)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1]))

def _train_one(seed):
    dl.set_seed(seed)
    Xtr, Ytr = pooled("train", markets); Xva, Yva = pooled("val", markets); y_sd = Ytr.std(0)
    model = Forecaster(); T = torch.tensor
    tr = DataLoader(TensorDataset(T(Xtr), T(Ytr / y_sd)), batch_size=32, shuffle=True)
    va = DataLoader(TensorDataset(T(Xva), T(Yva / y_sd)), batch_size=512)
    dl.fit(model, tr, va, nn.HuberLoss(), torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4), epochs=80, device=cpu,
           early_stopper=dl.EarlyStopper(patience=10), grad_clip=1.0, task="regression", verbose=False)
    model.eval()
    with torch.no_grad():
        return model(T(X_va)).numpy() * y_sd, model(T(X_te)).numpy() * y_sd

def forecast():
    runs = [_train_one(s) for s in (0, 1, 2)]                         # a 3-seed ensemble
    return {"val": np.mean([r[0] for r in runs], 0), "test": np.mean([r[1] for r in runs], 0)}

fc = forecast()

In [8]:
# ▶ self-check
ok = isinstance(fc, dict) and fc["test"].shape == Y_te.shape and fc["val"].shape == Y_va.shape
dl.check("returns val and test predictions with the right shapes", ok)
if ok:
    actual = to_kes(origins_te, Y_te)[:, 3]
    mae = lambda d: np.abs(to_kes(origins_te, d)[:, 3] - actual).mean()
    rel = mae(fc["test"]) / mae(np.zeros_like(Y_te))
    print(f"4-week test MAE: {mae(fc['test']):.0f} KES vs naive {mae(np.zeros_like(Y_te)):.0f} KES → relative MAE {rel:.2f}")
    dl.check(f"beats naive: relative MAE ≤ 0.8  (yours: {rel:.2f})", rel <= 0.8, "try Ridge on the flattened windows, or an LSTM with rain+fuel inputs")

✅ returns val and test predictions with the right shapes
4-week test MAE: 240 KES vs naive 427 KES → relative MAE 0.56
✅ beats naive: relative MAE ≤ 0.8  (yours: 0.56)


**Why this works.** Rainfall and fuel (with their lags) carry real signal; a model that sees them, and is regularised/ensembled for tiny data, beats 'no change' by a wide margin.

---
### 🧩 Problem 16.4: Milestone 4 - The advice engine  &nbsp; `⭐⭐⭐`

Write `advice_card(leaf_probs, class_names, price_now, delta_pred, delta_lo, delta_hi, storage_cost=0.03)` returning a dict. Rules:

**Diagnosis tier** by `p = max(leaf_probs)`: `p ≥ 0.90` → `"diagnose"`; `0.60 ≤ p < 0.90` → `"retake"`; `p < 0.60` → `"refer"`.
* `diagnose`: `card["diagnosis"]` is the predicted class name; `card["diagnosis_text"]` = the matching entry of `DISEASE_ADVICE`.
* `retake`: `card["diagnosis"] = None`; text says to retake the photo and names the **top-2 guesses with percentages**.
* `refer`: `card["diagnosis"] = None`; text asks the farmer to consult the extension officer.

**Price outlook** (4 weeks ahead): `price_forecast = price_now·exp(delta_pred)`, `price_low = price_now·exp(delta_lo)`, `price_high = price_now·exp(delta_hi)`; `card["price_text"]` is a readable sentence containing the rounded forecast and range.

**Action** (`gain = price_forecast/price_now − 1`):
* `gain > storage_cost` **and** `price_low > price_now·(1+storage_cost)` → `"HOLD"`
* `gain > storage_cost` but the lower bound isn't above → `"HOLD (low confidence)"`
* otherwise → `"SELL NOW"`

Include all keys: `tier, diagnosis, diagnosis_text, price_now, price_forecast, price_low, price_high, price_text, action`.


In [9]:
DISEASE_ADVICE = {
    "healthy": "No disease detected. Keep scouting weekly.",
    "gray_leaf_spot": "Looks like GRAY LEAF SPOT. Improve air flow, rotate crops, consider fungicide if spreading; confirm with your extension officer.",
    "northern_leaf_blight": "Looks like NORTHERN LEAF BLIGHT. Use resistant varieties next season; fungicide may help early; confirm with your extension officer.",
    "common_rust": "Looks like COMMON RUST. Usually mild; monitor and spray only if severe; confirm with your extension officer.",
    "maize_streak": "Looks like MAIZE STREAK VIRUS. No cure: remove and destroy affected plants, control leafhoppers, use resistant varieties; confirm with your extension officer.",
}

def advice_card(leaf_probs, class_names, price_now, delta_pred, delta_lo, delta_hi, storage_cost=0.03):
    leaf_probs = np.asarray(leaf_probs); p = float(leaf_probs.max()); top = int(leaf_probs.argmax())
    card = {}
    if p >= 0.90:
        card.update(tier="diagnose", diagnosis=class_names[top], diagnosis_text=DISEASE_ADVICE[class_names[top]])
    elif p >= 0.60:
        a, b = np.argsort(-leaf_probs)[:2]
        card.update(tier="retake", diagnosis=None, diagnosis_text=f"Not sure. Please retake the photo (good light, one leaf, fill the frame). Top guesses: {class_names[a]} ({leaf_probs[a]:.0%}), {class_names[b]} ({leaf_probs[b]:.0%}).")
    else:
        card.update(tier="refer", diagnosis=None, diagnosis_text="I cannot tell from this photo. Please consult your extension officer.")
    fc_, lo_, hi_ = (price_now * np.exp(d) for d in (delta_pred, delta_lo, delta_hi))
    card.update(price_now=price_now, price_forecast=fc_, price_low=lo_, price_high=hi_,
                price_text=f"Today {price_now:,.0f} KES/bag. In 4 weeks expect about {fc_:,.0f} (likely range {lo_:,.0f}–{hi_:,.0f}).")
    gain = fc_ / price_now - 1
    if gain > storage_cost and lo_ > price_now * (1 + storage_cost):
        card["action"] = "HOLD"
    elif gain > storage_cost:
        card["action"] = "HOLD (low confidence)"
    else:
        card["action"] = "SELL NOW"
    return card

In [10]:
# ▶ self-check
names = dl.LEAF_CLASSES
try:
    c1 = advice_card(np.array([.02, .95, .01, .01, .01]), names, 4800, np.log(1.08), np.log(1.05), np.log(1.12))
    dl.check("confident diagnosis → tier 'diagnose' with the class name", c1["tier"] == "diagnose" and c1["diagnosis"] == "gray_leaf_spot")
    dl.check("diagnosis text comes from DISEASE_ADVICE", c1["diagnosis_text"] == DISEASE_ADVICE["gray_leaf_spot"])
    c2 = advice_card(np.array([.70, .25, .02, .02, .01]), names, 4800, 0, 0, 0)
    dl.check("0.6 ≤ p < 0.9 → 'retake', no diagnosis, names the top-2 guesses", c2["tier"] == "retake" and c2["diagnosis"] is None and "healthy" in c2["diagnosis_text"] and "gray_leaf_spot" in c2["diagnosis_text"])
    c3 = advice_card(np.array([.40, .30, .15, .10, .05]), names, 4800, 0, 0, 0)
    dl.check("p < 0.6 → 'refer' and NO diagnosis (never confident when unsure)", c3["tier"] == "refer" and c3["diagnosis"] is None)
    dl.check("price numbers: forecast = price_now·exp(Δ)", np.isclose(c1["price_forecast"], 4800 * 1.08) and np.isclose(c1["price_low"], 4800 * 1.05) and np.isclose(c1["price_high"], 4800 * 1.12))
    dl.check("+8% forecast, lower bound above the cost threshold → HOLD", c1["action"] == "HOLD")
    c4 = advice_card(np.array([.97, .01, .01, .005, .005]), names, 4800, np.log(1.08), np.log(0.97), np.log(1.15))
    dl.check("+8% but wide interval → HOLD (low confidence)", c4["action"] == "HOLD (low confidence)")
    c5 = advice_card(np.array([.97, .01, .01, .005, .005]), names, 4800, np.log(1.01), np.log(0.98), np.log(1.04))
    dl.check("+1% (< storage cost) → SELL NOW", c5["action"] == "SELL NOW")
    dl.check("all required keys are present", all(k in c1 for k in ("tier", "diagnosis", "diagnosis_text", "price_now", "price_forecast", "price_low", "price_high", "price_text", "action")))
except Exception as e:
    dl.check("advice_card runs on the test cases", False, f"{type(e).__name__}: {e}")

✅ confident diagnosis → tier 'diagnose' with the class name
✅ diagnosis text comes from DISEASE_ADVICE
✅ 0.6 ≤ p < 0.9 → 'retake', no diagnosis, names the top-2 guesses
✅ p < 0.6 → 'refer' and NO diagnosis (never confident when unsure)
✅ price numbers: forecast = price_now·exp(Δ)
✅ +8% forecast, lower bound above the cost threshold → HOLD
✅ +8% but wide interval → HOLD (low confidence)
✅ +1% (< storage cost) → SELL NOW
✅ all required keys are present


**Why this works.** Notice the design principle: the system **abstains** when unsure (retake/refer) and labels its own low-confidence recommendations. Knowing when *not* to answer is a feature.

---
### 🧩 Problem 16.5: Milestone 5 - Evaluate the whole system  &nbsp; `⭐⭐`

Write `system_report(leaf_model, leaf, fc)` returning a dict with:
`test_acc`, `per_class_recall` (list of 5), `mae_4w`, `naive_mae_4w`, `rel_mae_4w` (all in KES, on the **test** set, 4-week horizon),
`q_lo`, `q_hi` (the 10th and 90th percentiles of **validation** residuals `Y_va − fc["val"]`, per horizon; array of 4) and `coverage_4w` (fraction of test 4-week log-changes falling inside `[pred+q_lo[3], pred+q_hi[3]]`).
Then call `report = system_report(leaf_model, leaf, fc)` and **read the numbers critically**.


In [11]:
def system_report(leaf_model, leaf, fc):
    probs = dl.predict_proba(leaf_model, leaf["X_te"], device=cpu).numpy(); preds = probs.argmax(1); y = leaf["y_te"]
    recall = [float((preds[y == c] == c).mean()) for c in range(5)]
    actual = to_kes(origins_te, Y_te)[:, 3]
    mae = lambda d: float(np.abs(to_kes(origins_te, d)[:, 3] - actual).mean())
    q_lo, q_hi = np.quantile(Y_va - fc["val"], 0.10, axis=0), np.quantile(Y_va - fc["val"], 0.90, axis=0)
    p4 = fc["test"][:, 3]
    coverage = float(((Y_te[:, 3] >= p4 + q_lo[3]) & (Y_te[:, 3] <= p4 + q_hi[3])).mean())
    return dict(test_acc=float((preds == y).mean()), per_class_recall=recall, mae_4w=mae(fc["test"]), naive_mae_4w=mae(np.zeros_like(Y_te)),
                rel_mae_4w=mae(fc["test"]) / mae(np.zeros_like(Y_te)), q_lo=q_lo, q_hi=q_hi, coverage_4w=coverage)

report = system_report(leaf_model, leaf, fc)

In [12]:
# ▶ self-check
ok = isinstance(report, dict) and all(k in report for k in ("test_acc", "per_class_recall", "mae_4w", "naive_mae_4w", "rel_mae_4w", "q_lo", "q_hi", "coverage_4w"))
dl.check("report has all the required keys", ok)
if ok:
    print(f"test acc {report['test_acc']:.1%} | recalls {np.round(report['per_class_recall'], 2)} | 4w MAE {report['mae_4w']:.0f} vs naive {report['naive_mae_4w']:.0f} (rel {report['rel_mae_4w']:.2f}) | 80% interval coverage {report['coverage_4w']:.0%}")
    dl.check("test accuracy ≥ 85%", report["test_acc"] >= 0.85)
    dl.check("worst-class recall ≥ 65%", min(report["per_class_recall"]) >= 0.65)
    dl.check("price forecast beats 0.8 × naive", report["rel_mae_4w"] <= 0.8)
    dl.check("interval coverage between 65% and 92% (nominal 80%)", 0.65 <= report["coverage_4w"] <= 0.92)
    dl.check("relative MAE is consistent with MAE / naive MAE", np.isclose(report["rel_mae_4w"], report["mae_4w"] / report["naive_mae_4w"]))

✅ report has all the required keys
test acc 91.1% | recalls [0.92 0.91 0.86 1.   0.88] | 4w MAE 240 vs naive 427 (rel 0.56) | 80% interval coverage 84%
✅ test accuracy ≥ 85%
✅ worst-class recall ≥ 65%
✅ price forecast beats 0.8 × naive
✅ interval coverage between 65% and 92% (nominal 80%)
✅ relative MAE is consistent with MAE / naive MAE


**Why this works.** Calibrating the intervals on *validation* residuals and then checking coverage on *test* is exactly the honest protocol: the test set plays no role in building the system.

---
### 🧩 Problem 16.6: Milestone 6 - The model card  &nbsp; `⭐⭐`

A **model card** (Mitchell et al., 2019) is a short document that travels with a model. Write `model_card`: a Markdown **string**: with these sections (use `## ` headings):
**Intended use**, **Data**, **Metrics**, **Limitations**, **Ethical considerations**.
It must contain your **actual numbers** (use the `report` dict and f-strings), at least **three specific limitations**, and **no leftover "TODO"** text. Aim for 200–400 words. A fuller template is in `../capstone/MODEL_CARD_TEMPLATE.md`.


In [13]:
model_card = f"""
# FieldAdvisor model card

## Intended use
A *decision-support prototype* for smallholder maize farmers: it suggests which of five leaf conditions a photo shows and gives a 4-week price outlook for Nairobi. It is **not** a replacement for an agronomist and must not be used for high-stakes decisions without human review.

## Data
Synthetic 48×48 RGB maize-leaf images (2,100 train / 450 val / 450 test; 5 classes) and synthetic weekly maize prices for Nairobi, Eldoret and Kisumu 2011–2025 (train ≤ 2020, validation 2021–22, test 2023–25). Leaf statistics are computed on the training split only.

## Metrics
- Disease classifier (test): accuracy **{report['test_acc']:.1%}**; per-class recall {', '.join(f"{n} {r:.0%}" for n, r in zip(leaf['class_names'], report['per_class_recall']))}.
- Price forecaster (4 weeks ahead, test): MAE **{report['mae_4w']:,.0f} KES** vs {report['naive_mae_4w']:,.0f} KES for the naive 'no change' forecast (relative MAE {report['rel_mae_4w']:.2f}).
- 80% prediction interval: empirical test coverage **{report['coverage_4w']:.0%}**.

## Limitations
1. **Synthetic data:** real leaves have messy backgrounds, lighting and co-infections; performance on field photos is unknown and likely lower.
2. **Mild/early infections** are the hardest cases for the classifier, and they are the most valuable to catch.
3. **Small test sets** (450 images, ~150 overlapping price windows) give wide uncertainty on every number above.
4. Prices follow simulated drivers (rainfall, fuel, season); real markets also react to policy, imports and shocks the model has never seen.
5. Interval coverage assumes the future is about as volatile as the validation period.

## Ethical considerations
Wrong advice can cost a harvest or a sale, so the app **abstains** when unsure (retake/refer) and always recommends confirming with an extension officer. Farmers without smartphones or reliable connectivity are excluded unless the model runs offline; advice should be available in Swahili and local languages. Real photos must be collected with informed consent and fair benefit-sharing, and deployment needs monitoring for drift (new varieties, new diseases, new phones).
"""

In [14]:
# ▶ self-check
txt = model_card if "model_card" in globals() and isinstance(model_card, str) else ""
low = txt.lower()
dl.check("has the five required sections", all(s in low for s in ("## intended use", "## data", "## metrics", "## limitations", "## ethical considerations")))
dl.check("no leftover TODO", "todo" not in low and len(txt) > 0)
dl.check("substantive length (≥ 800 characters)", len(txt) >= 800)
dl.check("contains real numbers (percentages and KES)", txt.count("%") >= 3 and "kes" in low)
lim = low.split("## limitations")[-1].split("## ethical")[0] if "## limitations" in low else ""
dl.check("lists at least three limitations", sum(1 for ln in lim.splitlines() if ln.strip()[:2] in ("1.", "2.", "3.", "- ", "* ")) >= 3)

✅ has the five required sections
✅ no leftover TODO
✅ substantive length (≥ 800 characters)
✅ contains real numbers (percentages and KES)
✅ lists at least three limitations


---
### 🧩 Problem 16.7: Milestone 7 - End-to-end demo  &nbsp; `⭐⭐`

Write `render_demo(leaf_model, leaf, fc, report, n=3)` returning a list of `n` advice cards (dicts from `advice_card`), each built from **one test leaf image and one test price origin**:
pick `n` random test leaves and `n` random test origins with `rng = np.random.default_rng(1)` (`rng.choice(..., n, replace=False)`), use `price_now = series["Nairobi"]["price"][origin]`,
the 4-week forecast `fc["test"][i, 3]` and its interval `fc["test"][i, 3] + report["q_lo"][3]` / `+ report["q_hi"][3]`.
Then print each card with `format_card`.


In [15]:
def render_demo(leaf_model, leaf, fc, report, n=3):
    probs = dl.predict_proba(leaf_model, leaf["X_te"], device=cpu).numpy()
    rng = np.random.default_rng(1)
    leaf_idx = rng.choice(len(probs), n, replace=False); price_idx = rng.choice(len(origins_te), n, replace=False)
    cards = []
    for li, pi in zip(leaf_idx, price_idx):
        d = fc["test"][pi, 3]; price_now = series["Nairobi"]["price"][origins_te[pi]]
        cards.append(advice_card(probs[li], leaf["class_names"], price_now, d, d + report["q_lo"][3], d + report["q_hi"][3]))
    return cards

cards = render_demo(leaf_model, leaf, fc, report)
for c in cards:
    print(format_card(c))

┌──────────────────────────────────────────────────────────────────┐
│ 🌽 FIELD ADVISOR                                                  │
├──────────────────────────────────────────────────────────────────┤
│ 🍃 Leaf: I cannot tell from this photo. Please consult your e     │
│       : xtension officer.                                        │
│ 📈 Price: Today 7,031 KES/bag. In 4 weeks expect about 7,283 (    │
│        : likely range 6,804–7,838).                              │
│ ✅ Action: HOLD (low confidence)                                  │
└──────────────────────────────────────────────────────────────────┘
┌──────────────────────────────────────────────────────────────────┐
│ 🌽 FIELD ADVISOR                                                  │
├──────────────────────────────────────────────────────────────────┤
│ 🍃 Leaf: Looks like MAIZE STREAK VIRUS. No cure: remove and d     │
│       : estroy affected plants, control leafhoppers, use res     │
│       : istant varieties; confir

In [16]:
# ▶ self-check
ok = "cards" in globals() and isinstance(cards, list) and len(cards) == 3
dl.check("three cards produced", ok)
dl.check("each card has a tier, diagnosis text, price text and action", ok and all(all(k in c for k in ("tier", "diagnosis_text", "price_text", "action")) for c in cards))
dl.check("actions are valid", ok and all(c["action"] in ("HOLD", "HOLD (low confidence)", "SELL NOW") for c in cards))
dl.check("no confident diagnosis below the threshold", ok and all(c["tier"] == "diagnose" or c["diagnosis"] is None for c in cards))

✅ three cards produced
✅ each card has a tier, diagnosis text, price text and action
✅ actions are valid
✅ no confident diagnosis below the threshold


**Why this works.** This is the whole pipeline in one screen: two models, uncertainty, abstention and a business rule: all driven by *test* data that no model was tuned on.